# Training & Validation

**All planes (axial / coronal / sagittal) x all tasks (abnormal, ACL tear, meniscus tear).**

Per `explanation.txt`: per-plane multi-task model — **3 sigmoid outputs per exam**,
P(abnormal), P(ACL tear), P(meniscal tear) — then the exam-level probability is the
**average across the 3 plane models**. Threshold: P(task) > 0.5 -> positive.

HPO (Grid / Randomized / Bayesian-Optuna) -> full-epoch training with early stopping
-> validation -> joblib artifacts (built from this notebook only).
No backend / API — clean model training and validation only.

In [ ]:
import os
import copy
import gc
import time

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as Fn
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tvm
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
from sklearn.model_selection import ParameterGrid, ParameterSampler
from scipy.stats import loguniform, uniform
import optuna

SEED = 42
IMG_DIR = '/kaggle/input/knee-mri/images'   # Kaggle dataset mount point

# Discover the correct Kaggle mount path for the knee-mri dataset
import glob as _glob
_candidates = [
    '/kaggle/input/datasets/prinshudewangan/knee-mri/images',
    '/kaggle/input/datasets/prinshudewangan/knee-mri',
    '/kaggle/input/knee-mri/images',
    '/kaggle/input/kneemri/images',
    '/kaggle/input/prinshudewangan/knee-mri/images',
    '/kaggle/input/knee-mri',
    '/kaggle/input/kneemri',
    '/kaggle/input/prinshudewangan/knee-mri',
]
_found = None
for _c in _candidates:
    if _glob.glob(_c + '/train-abnormal.csv'):
        _found = _c
        break
if _found is None:
    print('Searching /kaggle/input/ for knee-mri data...')
    for _p in _glob.glob('/kaggle/input/*'):
        print('  ', _p)
        for _sub in _glob.glob(_p + '/*'):
            print('    ', _sub)
    raise FileNotFoundError('Could not find knee-mri dataset images directory')
IMG_DIR = _found
print(f'Using IMG_DIR: {IMG_DIR}')
PLANES = ('axial', 'coronal', 'sagittal')   # every plane is trained
TASKS = ('abnormal', 'acl', 'meniscus')     # every task label is trained
BATCH_SIZE = 8                              # was 1 — 4 keeps GPU busy without OOM on T4
LR = 5e-4
EPOCHS = 100
PATIENCE = 20
BACKBONE = 'resnet18'
OUT_DIR = 'outputs'
MODE = 'all'   # full dataset: all planes, all tasks, no limits
# Reduced HPO budget so the whole notebook finishes inside one Kaggle session (~2.5h).
# Original was {'grid': 12, 'random': 10, 'optuna': 15, 'epochs': 50, ...} → ~27h.
HPO_BUDGET = {'demo':   {'grid': 6, 'random': 8,  'optuna': 10, 'epochs': 5,  'train_n': 48,  'val_n': 24},
              'single': {'grid': 6, 'random': 6, 'optuna': 10, 'epochs': 8,  'train_n': 48,  'val_n': 24},
              'all':    {'grid': 12, 'random': 24, 'optuna': 32, 'epochs': 30, 'train_n': 192,  'val_n': 96}}[MODE]
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
AMP = DEVICE == 'cuda'

torch.backends.cudnn.benchmark = True
os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)
print(f"device: {DEVICE} | AMP: {AMP} | mode: {MODE} | hpo budget (per plane): {HPO_BUDGET}")

Preprocessing, augmentation, dataset.

The dataset merges the **3 task CSVs** (`{split}-abnormal.csv`, `{split}-acl.csv`,
`{split}-meniscus.csv`) by exam id, so one sample carries the full label vector
`(abnormal, acl, meniscus)` for one plane.

In [ ]:
def standardize(arr):
    arr = arr.astype(np.float32)
    return (arr - arr.mean()) / (arr.std() + 1e-8)


def preprocess(arr, target=32):
    arr = standardize(arr)
    t = torch.from_numpy(np.ascontiguousarray(arr))[None, None]
    arr = Fn.interpolate(t, size=(target, arr.shape[1], arr.shape[2]),
                         mode='trilinear', align_corners=False)[0, 0].numpy()
    return torch.from_numpy(np.stack([arr] * 3))


def load_volume(path, target=32):
    """Load one exam/plane .npy -> (3, 32, H, W) tensor."""
    return preprocess(np.load(path), target)

In [ ]:
def _slice_batch(vol):
    d, h, w = vol.shape[1:]
    return vol.permute(1, 0, 2, 3).reshape(-1, 1, h, w)


def _unslice_batch(imgs, c, d):
    h, w = imgs.shape[-2:]
    return imgs.reshape(d, c, h, w).permute(1, 0, 2, 3)


class RandomHFlip:
    def __init__(self, p=0.5):
        self.p = p

    def __call__(self, vol):
        return torch.flip(vol, dims=[-1]) if torch.rand(1).item() < self.p else vol


class RandomRotation:
    def __init__(self, degrees=15.0, p=0.5):
        self.degrees, self.p = degrees, p

    def __call__(self, vol):
        if torch.rand(1).item() >= self.p or self.degrees == 0:
            return vol
        ang = (torch.rand(1).item() * 2 - 1) * self.degrees
        imgs = TF.rotate(_slice_batch(vol), ang, interpolation=InterpolationMode.BILINEAR, fill=0.0)
        return _unslice_batch(imgs, vol.shape[0], vol.shape[1])


class RandomTranslation:
    def __init__(self, fraction=0.1, p=0.5):
        self.fraction, self.p = fraction, p

    def __call__(self, vol):
        if torch.rand(1).item() >= self.p or self.fraction == 0:
            return vol
        dx = int(round((torch.rand(1).item() * 2 - 1) * self.fraction * vol.shape[-1]))
        dy = int(round((torch.rand(1).item() * 2 - 1) * self.fraction * vol.shape[-2]))
        imgs = TF.affine(_slice_batch(vol), angle=0, translate=(dx, dy), scale=1.0,
                         shear=[0.0, 0.0], interpolation=InterpolationMode.BILINEAR, fill=0.0)
        return _unslice_batch(imgs, vol.shape[0], vol.shape[1])


def make_transforms(degrees=15.0, translation=0.1):
    return RandomHFlip(0.5), RandomRotation(degrees, 0.5), RandomTranslation(translation, 0.5)


class ComposeTransforms:
    """Picklable replacement for a lambda so DataLoader workers can spawn."""

    def __init__(self, hflip, rot, trans):
        self.hflip, self.rot, self.trans = hflip, rot, trans

    def __call__(self, vol):
        return self.trans(self.rot(self.hflip(vol)))


class MRNetDataset(Dataset):
    """One plane, ALL tasks: 3 label CSVs merged per exam -> y = (abnormal, acl, meniscus)."""

    def __init__(self, root=IMG_DIR, split='train', plane='sagittal',
                 degrees=15.0, translation=0.1, augment=False):
        self.folder = os.path.join(root, split, plane)
        merged = None
        for task in TASKS:
            df = pd.read_csv(os.path.join(root, f'{split}-{task}.csv'),
                             header=None, names=['id', task])
            df['id'] = df['id'].astype(int).astype(str).str.zfill(4)
            df = df.set_index('id')
            merged = df if merged is None else merged.join(df, how='inner')
        merged = merged[list(TASKS)].reset_index()
        self.ids = merged['id'].tolist()
        self.labels = merged[list(TASKS)].to_numpy(np.float32)          # (N, 3)
        hflip, rot, trans = make_transforms(degrees, translation)
        self.transform = ComposeTransforms(hflip, rot, trans) if augment else None
        pos = self.labels.sum(axis=0)
        neg = len(self) - pos
        self.pos_weights = torch.tensor(neg / np.maximum(pos, 1.0))     # (3,)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        # Load lazily. A per-worker cache of all 1130 exams is ~28 GB per
        # worker (3 x 32 x 256 x 256 float32 ~= 25 MB each), which gets the
        # DataLoader worker OOM-killed. Each epoch visits every exam once,
        # so caching gave no speedup anyway.
        vol = load_volume(os.path.join(self.folder, self.ids[idx] + '.npy'))
        if self.transform is not None:
            vol = self.transform(vol)
        return vol, torch.from_numpy(self.labels[idx])


Model: one multi-task head per plane (3 sigmoid outputs) + portable joblib payload
(state dict + config). Per-plane `.joblib` plus a single combined bundle joblib.

In [ ]:
class MRNet2D(nn.Module):
    """Per-plane multi-task backbone -> 3 logits: (abnormal, acl, meniscus)."""

    def __init__(self, backbone='resnet18', pretrained=True, n_tasks=len(TASKS)):
        super().__init__()
        if backbone == 'resnet18':
            m = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
            m.fc = nn.Identity()
            self.backbone, self.feat_dim = nn.Sequential(*list(m.children())), 512
        elif backbone == 'alexnet':
            m = tvm.alexnet(weights=tvm.AlexNet_Weights.IMAGENET1K_V1 if pretrained else None)
            self.backbone, self.feat_dim = nn.Sequential(m.features, nn.AdaptiveAvgPool2d(1), nn.Flatten()), 256
        else:
            raise ValueError(backbone)
        self.fc = nn.Linear(2 * self.feat_dim, n_tasks)

    def forward(self, x):                     # (B, 3, D, H, W)
        b, c, d, h, w = x.shape
        f = self.backbone(x.permute(0, 2, 1, 3, 4).reshape(b * d, c, h, w)).view(b, d, self.feat_dim)
        return self.fc(torch.cat([f.max(1).values, f.logsumexp(1)], dim=1))   # (B, n_tasks)


class MRNetPipeline:
    """Inference wrapper for one plane's multi-task model."""

    def __init__(self, plane, model, thresholds=None, device=None, amp=True):
        self.plane = plane
        self.thresholds = thresholds or {t: 0.5 for t in TASKS}
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.amp = self.device == 'cuda' and amp
        self.model = model.to(self.device).eval()

    def warm(self, device=None):
        if device:
            self.device = device
            self.model = self.model.to(device).eval()
        self.amp = self.device == 'cuda' and self.amp
        x = torch.randn(1, 3, 32, 256, 256, device=self.device)
        with torch.inference_mode():
            self.model(x)
        return self

    @torch.inference_mode()
    def predict_proba(self, vol):
        if not torch.is_tensor(vol):
            vol = load_volume(vol)
        x = vol.unsqueeze(0).to(self.device)
        with torch.amp.autocast('cuda', enabled=self.amp):
            p = torch.sigmoid(self.model(x))[0].float()
        return {t: float(p[j]) for j, t in enumerate(TASKS)}

    def predict(self, vol):
        proba = self.predict_proba(vol)
        return {t: {'probability': round(p, 4), 'label': int(p >= self.thresholds[t])}
                for t, p in proba.items()}


def _state_cpu(model):
    return {k: v.detach().cpu() for k, v in model.state_dict().items()}


def save_pipeline(model, plane, out_dir=OUT_DIR, name=None):
    path = os.path.join(out_dir, name or f'model_{plane}.joblib')
    payload = {'backbone': BACKBONE, 'plane': plane, 'tasks': TASKS,
               'thresholds': {t: 0.5 for t in TASKS}, 'model_state': _state_cpu(model)}
    joblib.dump(payload, path, compress=3)
    return path


def load_pipeline(path):
    payload = joblib.load(path)
    model = MRNet2D(payload['backbone'], pretrained=False)
    model.load_state_dict(payload['model_state'])
    return MRNetPipeline(payload['plane'], model, thresholds=payload['thresholds']).warm()


def save_bundle(models, out_dir=OUT_DIR, name=None):
    """ONE joblib holding every plane's model - the single artifact built from this notebook."""
    path = os.path.join(out_dir, name or 'knee_mri_multitask.joblib')
    payload = {'backbone': BACKBONE, 'tasks': TASKS, 'planes': PLANES,
               'thresholds': {t: 0.5 for t in TASKS},
               'models': {pl: _state_cpu(m) for pl, m in models.items()}}
    joblib.dump(payload, path, compress=3)
    return path


def load_bundle(path):
    payload = joblib.load(path)
    pipes = {}
    for pl, state in payload['models'].items():
        m = MRNet2D(payload['backbone'], pretrained=False)
        m.load_state_dict(state)
        pipes[pl] = MRNetPipeline(pl, m, thresholds=payload['thresholds']).warm()
    return pipes


BUNDLE_PIPES = None


def predict_exam(exam_id, split='valid', pipes=None):
    """Exam-level prediction: run all 3 plane models, average P(task) across planes."""
    pipes = pipes or BUNDLE_PIPES
    per_plane = {pl: pipes[pl].predict_proba(os.path.join(IMG_DIR, split, pl, f'{exam_id}.npy'))
                 for pl in PLANES}
    fused = {t: float(np.mean([per_plane[pl][t] for pl in PLANES])) for t in TASKS}
    return {'exam': exam_id, 'per_plane': per_plane,
            'fused': {t: round(p, 4) for t, p in fused.items()},
            'labels': {t: int(fused[t] >= 0.5) for t in TASKS}}

In [ ]:
model = MRNet2D(BACKBONE).to(DEVICE)
print('params_M:', round(sum(p.numel() for p in model.parameters()) / 1e6, 2), '| on:', next(model.parameters()).device)
probe = MRNetDataset(split='train', plane='sagittal', augment=True)
print('train pos_weights:', dict(zip(TASKS, [round(w, 3) for w in probe.pos_weights.tolist()])))
print('positives:', {t: int(probe.labels[:, j].sum()) for j, t in enumerate(TASKS)},
      f'/ {len(probe)} exams per plane')

Loop: epochs -> batches -> task-weighted BCE (3 outputs) -> AMP backward
-> validation (per-task AUC) + early stopping.

In [ ]:
def auc_score(labels, scores):
    y = np.asarray(labels, dtype=np.float64)
    r = pd.Series(np.asarray(scores, dtype=np.float64)).rank(method='average').values
    pos, neg = y.sum(), len(y) - y.sum()
    if pos == 0 or neg == 0:
        return float('nan')
    return float((r[y == 1].sum() - pos * (pos + 1) / 2) / (pos * neg))


def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.float().to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=AMP):
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total += loss.item() * len(y)
        n += len(y)
    return total / max(n, 1)


@torch.inference_mode()
def evaluate(model, loader, criterion):
    model.eval()
    total, n, ys, ps = 0.0, 0, [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.float().to(DEVICE)
        with torch.amp.autocast('cuda', enabled=AMP):
            logits = model(x)
        total += criterion(logits.float(), y).item() * len(y)
        n += len(y)
        ys.append(y.cpu())
        ps.append(torch.sigmoid(logits.float()).cpu())
    y, p = torch.cat(ys), torch.cat(ps)
    aucs = [auc_score(y[:, j], p[:, j]) for j in range(len(TASKS))]
    acc = float((((p >= 0.5).float() == y).float().mean()))
    return total / max(n, 1), aucs, acc

HPO per plane: exhaustive grid (`ParameterGrid`), randomized (`ParameterSampler` over
*scipy.stats* distributions), Bayesian TPE (Optuna) — objective = **mean val AUC of the 3 tasks**.

In [ ]:
# Grid uses fixed discrete values.
PARAM_SPACE = {
    'lr':              [5e-5, 1e-4, 3e-4, 1e-3, 3e-3],
    'weight_decay':    [0.0, 1e-5, 1e-4, 5e-4],
    'freeze_backbone': [False, True],
    'degrees':         [0.0, 10.0, 15.0, 25.0, 35.0],
    'translation':     [0.0, 0.1, 0.15],
}

# Random search now uses real continuous distributions (scipy.stats).
# Previously it was `(1e-5, 3e-3)` etc., which ParameterSampler treats as a
# 2-item discrete set — that's what caused the "total space of parameters 8
# is smaller than n_iter=10" warning.
RANDOM_SPACE = {
    'lr':           loguniform(1e-5, 1e-2),
    'weight_decay': loguniform(1e-6, 1e-1),
    'degrees':      uniform(0.0, 35.0),
}


class TrialRunner:
    def __init__(self, plane, train_n, val_n, epochs):
        rng = np.random.default_rng(SEED)
        tr = MRNetDataset(split='train', plane=plane)   # one construction
        va = MRNetDataset(split='valid', plane=plane)   # one construction
        self.train_ids = sorted(rng.choice(tr.ids, size=min(train_n, len(tr.ids)), replace=False).tolist())
        self.val_ids   = sorted(rng.choice(va.ids, size=min(val_n, len(va.ids)), replace=False).tolist())
        self.train = self._cache(tr, 'train', plane, self.train_ids)
        self.val   = self._cache(va, 'valid', plane, self.val_ids)
        self.epochs = epochs
        self.pw = tr.pos_weights.to(DEVICE)

    @staticmethod
    def _cache(ds, split, plane, ids):
        idx = {i: k for k, i in enumerate(ds.ids)}
        out = []
        for eid in ids:
            arr = np.load(os.path.join(IMG_DIR, split, plane, eid + '.npy'))
            out.append((preprocess(arr), torch.from_numpy(ds.labels[idx[eid]])))
        return out

    def run(self, params, epochs=None, verbose=True):
        epochs = epochs or self.epochs
        hflip, rot, trans = make_transforms(float(params['degrees']), float(params['translation']))

        def apply_t(vol):
            return trans(rot(hflip(vol)))

        model = MRNet2D(BACKBONE).to(DEVICE)
        if params.get('freeze_backbone'):
            for p in model.backbone.parameters():
                p.requires_grad = False
        criterion = nn.BCEWithLogitsLoss(pos_weight=self.pw)
        optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                                     lr=float(params['lr']), weight_decay=float(params['weight_decay']))
        scaler = torch.amp.GradScaler('cuda', enabled=AMP)
        order = list(range(len(self.train)))
        rng = np.random.default_rng(SEED)
        for _ in range(epochs):
            model.train()
            rng.shuffle(order)
            for i in order:
                x = apply_t(self.train[i][0]).unsqueeze(0).to(DEVICE)
                y = self.train[i][1].float().view(1, -1).to(DEVICE)
                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast('cuda', enabled=AMP):
                    loss = criterion(model(x), y)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        vl, va, _ = evaluate_pairs(model, self.val, criterion)
        score = float(np.nanmean(va))
        if verbose:
            print(f'  trial lr={float(params["lr"]):.2e} wd={float(params["weight_decay"]):.1e} '
                  f'deg={float(params["degrees"]):.1f} fb={params.get("freeze_backbone", False)} '
                  f'-> val_loss {vl:.4f} aucs {[round(a,3) for a in va]} mean_auc {score:.4f}', flush=True)
        return score


@torch.inference_mode()
def evaluate_pairs(model, pairs, criterion):
    model.eval()
    ys, ps, tot = [], [], 0.0
    for vol, y in pairs:
        x = vol.unsqueeze(0).to(DEVICE)
        with torch.amp.autocast('cuda', enabled=AMP):
            logits = model(x)
        tot += criterion(logits.float(), y.float().view(1, -1).to(DEVICE)).item()
        ps.append(torch.sigmoid(logits.float()).cpu())
        ys.append(y)
    y, p = torch.stack(ys).float(), torch.cat(ps)
    aucs = [auc_score(y[:, j], p[:, j]) for j in range(len(TASKS))]
    acc = float((((p >= 0.5).float() == y).float().mean()))
    return tot / len(pairs), aucs, acc


def run_hpo(plane, budget):
    runner = TrialRunner(plane, budget['train_n'], budget['val_n'], budget['epochs'])
    rows, t0 = [], time.time()

    # --- Grid ---
    grid = list(ParameterGrid({k: v for k, v in PARAM_SPACE.items()
                               if k in ('lr', 'weight_decay', 'degrees')}))[:budget['grid']]
    print(f'[{plane}] grid search: {len(grid)} configs', flush=True)
    for i, p in enumerate(grid, 1):
        p = dict(p, freeze_backbone=False, translation=0.1)
        score = runner.run(p)
        rows.append({'method': 'grid', **p, 'val_auc': score})
        print(f'  [{plane}][grid {i}/{len(grid)}] done in {time.time()-t0:.0f}s total', flush=True)

    # --- Random ---
    sampler = list(ParameterSampler(RANDOM_SPACE, n_iter=budget['random'], random_state=SEED))
    print(f'[{plane}] random search: {len(sampler)} configs', flush=True)
    rng_fb = np.random.default_rng(SEED)
    for i, p in enumerate(sampler, 1):
        p = dict(lr=float(p['lr']), weight_decay=float(p['weight_decay']), degrees=float(p['degrees']),
                 freeze_backbone=bool(rng_fb.integers(2)), translation=0.1)
        score = runner.run(p)
        rows.append({'method': 'random', **p, 'val_auc': score})
        print(f'  [{plane}][random {i}/{len(sampler)}] done in {time.time()-t0:.0f}s total', flush=True)

    # --- Optuna ---
    def objective(trial):
        p = {'lr': trial.suggest_float('lr', 1e-5, 1e-2, log=True),
             'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-1, log=True),
             'degrees': trial.suggest_float('degrees', 0.0, 35.0),
             'freeze_backbone': trial.suggest_categorical('freeze_backbone', [False, True]),
             'translation': trial.suggest_categorical('translation', [0.0, 0.1])}
        return runner.run(p)

    print(f'[{plane}] optuna: {budget["optuna"]} trials', flush=True)
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=budget['optuna'], show_progress_bar=False)
    for p in study.trials:
        rows.append({'method': 'bayes', **p.params, 'val_auc': p.value})

    df = pd.DataFrame(rows).sort_values('val_auc', ascending=False).reset_index(drop=True)
    print(df.to_string(index=False))
    print(f'HPO done for {plane}: {len(df)} trials in {time.time() - t0:.0f}s', flush=True)
    best = df.iloc[0].to_dict()
    best = {k: (bool(v) if k == 'freeze_backbone' else float(v)) for k, v in best.items()
            if k not in ('method', 'val_auc')}
    print(f'best config for {plane}: {best} | val_auc={round(df.iloc[0]["val_auc"], 4)}', flush=True)
    del runner
    gc.collect()
    torch.cuda.empty_cache()
    return best, df


HPO for **every plane** (each plane's multi-task model gets its own tuned hyper-params).

In [ ]:
best_by_plane, hpo_tables = {}, {}
for plane in PLANES:
    print(f'=== HPO: plane={plane} (all 3 tasks) ===', flush=True)
    best_by_plane[plane], hpo_tables[plane] = run_hpo(plane, HPO_BUDGET)
    hpo_tables[plane].to_csv(os.path.join(OUT_DIR, f'hpo_{plane}_{MODE}.csv'), index=False)
    gc.collect()
    torch.cuda.empty_cache()
summary = pd.DataFrame([{'plane': pl, 'best_val_auc_mean': round(hpo_tables[pl]['val_auc'].max(), 4)}
                        for pl in PLANES])
print(summary.to_string(index=False))

In [ ]:
def train_model(plane, params=None, backbone=BACKBONE, epochs=None, patience=None,
                limit_train=None, limit_val=None, save_name=None, pretrained=True, tag=''):
    params = params or {}
    epochs, patience = epochs or EPOCHS, patience or PATIENCE
    train_ds = MRNetDataset(split='train', plane=plane, augment=True,
                            degrees=params.get('degrees', 15.0), translation=params.get('translation', 0.1))
    valid_ds = MRNetDataset(split='valid', plane=plane)
    if limit_train:
        idx = np.random.default_rng(SEED).choice(len(train_ds), size=min(limit_train, len(train_ds)), replace=False)
        train_ds = torch.utils.data.Subset(train_ds, idx.tolist())
    if limit_val:
        idx = np.random.default_rng(SEED).choice(len(valid_ds), size=min(limit_val, len(valid_ds)), replace=False)
        valid_ds = torch.utils.data.Subset(valid_ds, idx.tolist())
    g = torch.Generator().manual_seed(SEED)
    # num_workers=2 keeps augmentation off the main thread → 3–5× faster on Kaggle.
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True, persistent_workers=True,
                              prefetch_factor=2, generator=g)
    valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True, persistent_workers=True,
                              prefetch_factor=2)
    pos_weight = MRNetDataset(split='train', plane=plane).pos_weights.to(DEVICE)
    print(f'[{plane}] train={len(train_ds)} valid={len(valid_ds)} pos_weights='
          + str({t: round(w, 2) for t, w in zip(TASKS, pos_weight.tolist())}) + f' | params={params}',
          flush=True)

    model = MRNet2D(backbone, pretrained).to(DEVICE)
    if params.get('freeze_backbone'):
        for p in model.backbone.parameters():
            p.requires_grad = False
        print(f'frozen backbone -> trainable_M: '
              f'{sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.2f}')
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                                 lr=float(params.get('lr', LR)), weight_decay=float(params.get('weight_decay', 0.0)))
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    history = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_acc': []}
    best_val, best_state, wait = float('inf'), None, 0

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tl = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        vl, va, vac = evaluate(model, valid_loader, criterion)
        for k, v in zip(history, (tl, vl, va, vac)):
            history[k].append(v)
        flag = ''
        if vl < best_val:
            best_val, best_state, wait, flag = vl, copy.deepcopy(model.state_dict()), 0, ' *best*'
        else:
            wait += 1
        print(f'epoch {epoch:3d}/{epochs} | train {tl:.4f} | val {vl:.4f} | aucs '
              + str([round(a, 3) for a in va]) + f' | acc {vac:.3f} | {time.time() - t0:.1f}s{flag}', flush=True)
        if wait >= patience:
            print(f'early stopping at epoch {epoch} (no improvement for {patience} epochs)')
            break

    model.load_state_dict(best_state)
    save_name = save_name or f'best_{plane}{tag}.pth'
    pth_path = os.path.join(OUT_DIR, save_name)
    torch.save({'plane': plane, 'tasks': TASKS, 'params': params, 'epoch': len(history['val_loss']),
                'val_loss': best_val, 'model_state': best_state}, pth_path)
    jl_path = save_pipeline(model, plane,
                            name=os.path.basename(save_name).replace('best_', 'model_').rsplit('.', 1)[0] + '.joblib')
    print(f'saved {pth_path} + {jl_path} | best val_loss {best_val:.4f}', flush=True)
    return model, history, {'pth': pth_path, 'joblib': jl_path}

Train **all 3 planes** with their tuned params -> per-plane metrics, loss curves,
per-plane joblibs, and the single combined bundle joblib.

In [ ]:
RUN_FINAL = True
TAG = '' if MODE != 'demo' else '_demo'
if RUN_FINAL:
    demo = MODE == 'demo'
    models, histories, artifacts, rows = {}, {}, {}, []
    fig, axes = plt.subplots(1, len(PLANES), figsize=(6 * len(PLANES), 4))
    for ax, plane in zip(np.atleast_1d(axes), PLANES):
        model, history, art = train_model(plane, params=best_by_plane[plane], epochs=EPOCHS,
                                          patience=(EPOCHS + 1) if demo else PATIENCE,
                                          limit_train=HPO_BUDGET['train_n'] if demo else None,
                                          limit_val=HPO_BUDGET['val_n'] if demo else None,
                                          save_name=f'best_{plane}{TAG}.pth')
        models[plane], histories[plane], artifacts[plane] = model, history, art
        aucs = np.asarray(history['val_auc'], dtype=np.float64)
        row = {'plane': plane, 'mode': MODE, 'epochs_run': len(history['val_loss']),
               'best_val_loss': round(min(history['val_loss']), 4)}
        for j, t in enumerate(TASKS):
            row[f'auc_{t}'] = round(float(np.nanmax(aucs[:, j])), 4)
        row['auc_mean'] = round(float(np.nanmax(aucs.mean(axis=1))), 4)
        row.update({k: best_by_plane[plane][k] for k in ('lr', 'weight_decay', 'freeze_backbone',
                                                         'degrees', 'translation')})
        row['joblib'] = os.path.basename(art['joblib'])
        rows.append(row)
        ax.plot(range(1, len(history['val_loss']) + 1), history['val_loss'], marker='o', label='val')
        ax.plot(range(1, len(history['train_loss']) + 1), history['train_loss'], marker='.', label='train')
        ax.set_xlabel('epoch')
        ax.set_ylabel('weighted BCE (3 tasks)')
        ax.set_title(plane)
        ax.legend()
    results = pd.DataFrame(rows)
    results.to_csv(os.path.join(OUT_DIR, f'training_metrics{TAG or "_" + MODE}.csv'), index=False)
    print(results.to_string(index=False))
    fig.suptitle('early stopping on validation loss')
    fig.savefig(os.path.join(OUT_DIR, f'loss_curves{TAG or "_" + MODE}.png'), dpi=120, bbox_inches='tight')
    plt.show()

    bundle_path = save_bundle(models, name=f'knee_mri_multitask{TAG}.joblib')
    print('bundle:', bundle_path, f'({os.path.getsize(bundle_path) / 1e6:.1f} MB)')

Validation: reload the bundle joblib and predict whole exams —
per-plane probabilities + the fused (average across planes) exam-level result.

In [ ]:
bundle_path = os.path.join(OUT_DIR, f'knee_mri_multitask{TAG}.joblib')
print('loading', bundle_path, f'({os.path.getsize(bundle_path) / 1e6:.1f} MB)')
BUNDLE_PIPES = load_bundle(bundle_path)

out = predict_exam('1130')
print('exam 1130 ->', out)

t0 = time.time()
for i in range(5):
    out = predict_exam(f'{1130 + i}')
print(f'warm latency: {1000 * (time.time() - t0) / 5:.0f} ms/exam (all 3 planes, incl. preprocessing)')

## Artifacts (built from this notebook only)

- Per-plane: `outputs/model_{plane}{tag}.joblib` -> `load_pipeline()` gives one plane's
  multi-task predictor (`predict(vol)` -> {task: {probability, label}}).
- Single file: `outputs/knee_mri_multitask{tag}.joblib` -> `load_bundle()` +
  `predict_exam(id)` fuses the 3 planes by averaging -> exam-level
  P(abnormal), P(ACL tear), P(meniscal tear).

Decision rule (explanation.txt): P(task) > 0.5 -> task predicted.

Full sweep: `MRNET_MODE=all python -m jupyter nbconvert --to notebook --execute --inplace Training_and_validation.ipynb`

**Kaggle runtime note:** with the reduced HPO budget (`grid=6, random=8, optuna=12, epochs=15,
train_n=96, val_n=48`) and `BATCH_SIZE=4` + `num_workers=2`, HPO takes ~15–25 min per plane
(~1 h for 3 planes) and full training ~1.5 h — total ~2.5 h, well inside Kaggle's 9 h limit.